**UTS Data Mining II · Image**

Pilih satu input dan hapus sisanya sebelum Run All. Ganti path sesuai soal.

- Folder/ZIP: pakai **Grayscale dan resize**; hapus **Baca NPZ gambar**.
- NPZ: hapus input folder/ZIP serta **Grayscale dan resize**.

Lanjut ke **Tampilkan contoh gambar**.

In [ ]:
# Jalankan sekali jika diperlukan:
# %pip install numpy pandas matplotlib scikit-learn pillow openpyxl

**Import**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

**Baca folder gambar**

In [ ]:
data_dir = Path("/content/data/image")
extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
files = sorted([p for p in data_dir.rglob("*") if p.suffix.lower() in extensions])

df = pd.DataFrame({"path": [str(p) for p in files]})
print("Jumlah gambar:", len(df))
display(df.head())

**Baca ZIP gambar**

Ekstrak ke folder baru; subfolder ikut dibaca. Lanjut ke **Grayscale dan resize**.

In [ ]:
import zipfile

zip_path = "/content/image.zip"
data_dir = Path("/content/image_extracted")
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(data_dir)

extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
files = sorted([p for p in data_dir.rglob("*") if p.suffix.lower() in extensions])

df = pd.DataFrame({"path": [str(p) for p in files]})
print("Jumlah gambar:", len(df))
display(df.head())

**Grayscale dan resize**

Samakan gambar menjadi grayscale 28×28.

In [ ]:
images = []

for file in files:
    with Image.open(file) as img:
        img = img.convert("L").resize((28, 28))
        images.append(np.array(img))

X_images = np.array(images, dtype=np.float32) / 255.0
print("Shape gambar:", X_images.shape)
print("Rentang piksel:", X_images.min(), "sampai", X_images.max())

**Baca NPZ gambar**

`x_train`: grayscale 28×28, piksel 0–255; `y_train`: label. Contoh memakai 2.000 train (ubah `n`); test tetap terpisah. Label hanya untuk evaluasi. Normalisasi sudah dilakukan di loader; lanjut ke **Tampilkan contoh gambar**.

In [ ]:
data = np.load("/content/Data Latihan Datmin II (Image).npz", allow_pickle=False)
print("Nama array:", data.files)

n = 2000
X_images = data["x_train"][:n].astype(np.float32) / 255.0
df = pd.DataFrame({"label": data["y_train"][:n]})

print("Shape gambar:", X_images.shape)
display(df.head())

**Tampilkan contoh gambar**

In [ ]:
plt.figure(figsize=(8, 8))
for i in range(min(9, len(X_images))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_images[i], cmap="gray", vmin=0, vmax=1)
    plt.title(f"Gambar {i}")
    plt.axis("off")
plt.tight_layout()
plt.show()

**Flatten**

Satu gambar menjadi 784 fitur piksel.

In [ ]:
X = X_images.reshape(len(X_images), -1)
print("Shape fitur:", X.shape)

**Modeling dan evaluasi K-Means**

Coba beberapa `k`, masing-masing minimal 2 dan lebih kecil dari jumlah data. Silhouette memakai maksimal 3.000 data sebagai perkiraan skor.

In [ ]:
k_list = [3, 5, 7]
scores = []
inertia = []

for k in k_list:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    score = silhouette_score(X, labels, sample_size=min(3000, len(labels)), random_state=42)
    scores.append(score)
    inertia.append(model.inertia_)
    print(f"K={k}, silhouette={score:.4f}")

df_scores = pd.DataFrame({"k": k_list, "silhouette": scores, "inertia": inertia})
display(df_scores)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(k_list, scores, marker="o")
plt.xlabel("K")
plt.ylabel("Silhouette score")
plt.title("Silhouette")

plt.subplot(1, 2, 2)
plt.plot(k_list, inertia, marker="o")
plt.xlabel("K")
plt.ylabel("Inertia")
plt.title("Elbow")
plt.tight_layout()
plt.show()

**Model akhir**

Ganti `k = 7` sesuai hasil evaluasi atau soal.

In [ ]:
k = 7
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)
display(df.head())
display(df["cluster"].value_counts().sort_index())

**Visualisasi cluster dengan PCA**

PCA hanya untuk grafik; clustering tetap memakai fitur sebelumnya.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8, 6))
for cluster in range(k):
    mask = df["cluster"] == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f"Cluster {cluster}", alpha=0.6)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("Cluster gambar")
plt.legend()
plt.show()

**Centroid gambar**

Rata-rata piksel anggota tiap cluster.

In [ ]:
centers = kmeans.cluster_centers_.reshape(k, 28, 28)

plt.figure(figsize=(2 * k, 3))
for cluster in range(k):
    plt.subplot(1, k, cluster + 1)
    plt.imshow(centers[cluster], cmap="gray", vmin=0, vmax=1)
    plt.title(f"Cluster {cluster}")
    plt.axis("off")
plt.tight_layout()
plt.show()

**Contoh anggota tiap cluster**

In [ ]:
for cluster in range(k):
    indices = df.index[df["cluster"] == cluster][:3]
    plt.figure(figsize=(7, 3))
    for i, index in enumerate(indices):
        plt.subplot(1, 3, i + 1)
        plt.imshow(X_images[index], cmap="gray", vmin=0, vmax=1)
        plt.title(f"Gambar {index}")
        plt.axis("off")
    plt.suptitle(f"Cluster {cluster}")
    plt.tight_layout()
    plt.show()

**Simpan hasil (opsional)**

Hapus `#` pada kode untuk menyimpan CSV.

In [ ]:
# df.to_csv("image_hasil.csv", index=False)
# df_scores.to_csv("image_skor_k.csv", index=False)

**Alternatif input**
Pakai satu loader saja; hapus input lainnya sebelum Run All.

**Kaggle gambar** — install `kagglehub`; ganti `data_dir` pada loader folder.
```python
import kagglehub
folder = Path(kagglehub.dataset_download("pemilik/nama-dataset"))
print(list(folder.iterdir()))
data_dir = folder / "train"
```

**CSV lokasi gambar** — ganti loader folder. `filename` berisi path absolut atau relatif terhadap `data_dir`.
```python
data_dir = Path("/content/data/image")
df = pd.read_csv("/content/image_paths.csv")
files = [data_dir / str(name) for name in df["filename"]]
df["path"] = [str(p) for p in files]
df = df.reset_index(drop=True)
```
Excel: gunakan `pd.read_excel(...)`.

**CSV piksel** — ganti loader folder dan grayscale/resize. Contoh: 784 piksel 0–255 dan kolom `label`.
```python
df = pd.read_csv("/content/image_pixels.csv")
pixels = df.drop(columns=["label"]).to_numpy(dtype=np.float32)
X_images = pixels.reshape(-1, 28, 28) / 255.0
```
Keluarkan ID dari fitur. Jika piksel sudah 0–1, hapus pembagian 255.
Kaggle CSV piksel: gunakan `pd.read_csv(folder / "data.csv")`.

**MNIST** — install `tensorflow`; ganti loader folder dan grayscale/resize.
```python
from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_images = X_train[:2000].astype(np.float32) / 255.0
df = pd.DataFrame({"label": y_train[:2000]})
```
Contoh memakai 2.000 train; test tetap terpisah.

**Evaluasi label (opsional)**
Jalankan setelah **Model akhir**; sesuaikan kolom `label`.
```python
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
print("ARI:", adjusted_rand_score(df["label"], df["cluster"]))
print("NMI:", normalized_mutual_info_score(df["label"], df["cluster"]))
```
Periksa gambar anggota sebelum memberi arti pada cluster.